In [1]:
%pip install peft bitsandbytes

import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from transformers import BertTokenizerFast, BitsAndBytesConfig, BertForSequenceClassification, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

# -----------------------------------
# Traditional fine-tuning experiment
# -----------------------------------

# Load dataset
data = pd.read_csv("data/tweet_emotion_intensity/train.csv")

print(data.columns)  # use this to confirm column names

# Change these if your CSV uses different names
TEXT_COL = "tweet"
LABEL_COL = "labels"

# Split dataset
train_data, temp_data = train_test_split(
    data,
    test_size=0.3,
    random_state=42,
    stratify=data[LABEL_COL]
)

val_data, test_data = train_test_split(
    temp_data,
    test_size=0.5,
    random_state=42,
    stratify=temp_data[LABEL_COL]
)

# Reset indexes so iloc works cleanly
train_data = train_data.reset_index(drop=True)
val_data = val_data.reset_index(drop=True)
test_data = test_data.reset_index(drop=True)

print(f"Training set size: {len(train_data)}")
print(f"Validation set size: {len(val_data)}")
print(f"Test set size: {len(test_data)}")

# Tokenizer
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

# Dataset wrapper
class TweetDataset(torch.utils.data.Dataset):
    def __init__(self, df, tokenizer, text_col, label_col):
        self.df = df
        self.tokenizer = tokenizer
        self.text_col = text_col
        self.label_col = label_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        encoding = self.tokenizer(
            str(row[self.text_col]),
            truncation=True,
            padding="max_length",
            max_length=128
        )

        return {
            "input_ids": torch.tensor(encoding["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(encoding["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(int(row[self.label_col]), dtype=torch.long)
        }

train_dataset = TweetDataset(train_data, tokenizer, TEXT_COL, LABEL_COL)
val_dataset = TweetDataset(val_data, tokenizer, TEXT_COL, LABEL_COL)
test_dataset = TweetDataset(test_data, tokenizer, TEXT_COL, LABEL_COL)

# Model
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=12
)

# Training arguments
training_args = TrainingArguments(
    output_dir="./results_traditional",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    report_to="none"
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

# Train (The training and evaluation are commented because training this model without GPU support takes over 40 minutes)
# trainer.train()

# Evaluate
#results = trainer.evaluate(test_dataset)
#print(results)


# -----------------------------
# LoRA fine-tuning experiment
# -----------------------------
lora_model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=12
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query", "value"]
)

lora_model = get_peft_model(lora_model, lora_config)

lora_model.print_trainable_parameters()

lora_training_args = TrainingArguments(
    output_dir="./results_lora",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="no",
    logging_dir="./logs_lora",
    report_to="none"
)

lora_trainer = Trainer(
    model=lora_model,
    args=lora_training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

# Train (The training and evaluation are commented because training this model without GPU support takes over 30 minutes)
#lora_trainer.train()

# LoRA Results
#lora_results = lora_trainer.evaluate(test_dataset)
#print("LoRA fine-tuning results:")
#print(lora_results)


# -----------------------------
# QLoRA fine-tuning experiment
# -----------------------------

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True
)

qlora_model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=12,
    quantization_config=bnb_config,
    device_map="auto"
)

qlora_model = prepare_model_for_kbit_training(qlora_model)

qlora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query", "value"]
)

qlora_model = get_peft_model(qlora_model, qlora_config)

qlora_model.print_trainable_parameters()

qlora_training_args = TrainingArguments(
    output_dir="./results_qlora",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="no",
    logging_dir="./logs_qlora",
    report_to="none"
)

qlora_trainer = Trainer(
    model=qlora_model,
    args=qlora_training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

# Train (The training and evaluation are commented because qLoRA needs GPU support)
#qlora_trainer.train()

# Results
#qlora_results = qlora_trainer.evaluate(test_dataset)
#print("QLoRA fine-tuning results:")
#print(qlora_results)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 74.8 MB/s eta 0:00:00:00:0100:01

[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: /anaconda/envs/azureml_py310_sdkv2/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Index(['id', 'tweet', 'class', 'sentiment_intensity', 'class_intensity',
       'labels'],
      dtype='object')
Training set size: 2772
Validation set size: 594
Test set size: 594
trainable params: 304,140 || all params: 109,795,608 || trainable%: 0.2770


/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3695.67it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSI

AttributeError: 'Parameter' object has no attribute 'CB'